# Demo walkthrough

## 1. Project overview

This notebook demonstrates the minimal reproducible demo pipeline for unsupervised human motion analysis. It calls the existing project scripts, checks the included demo data, and displays generated CSV and PNG outputs without reimplementing the pipeline logic.

In [ ]:
from pathlib import Path
import subprocess
import sys

import pandas as pd
import matplotlib.pyplot as plt

try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value)


def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "demo_pipeline.py").exists() and (candidate / "Data").exists():
            return candidate
    raise FileNotFoundError("Could not find project root containing demo_pipeline.py and Data/.")


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "Data"
RESULTS_DIR = PROJECT_ROOT / "Results"


def rel(path):
    path = Path(path)
    try:
        return path.relative_to(PROJECT_ROOT).as_posix()
    except ValueError:
        return path.name


def sanitize_output(text):
    return (text or "").replace(str(PROJECT_ROOT), ".").strip()


def tail(text, lines=80):
    parts = text.splitlines()
    return "\n".join(parts[-lines:])


print("Project root detected.")

## 2. Check repository files

The demo expects the included data folder, the main pipeline script, dependency list, and Windows run scripts to be present.

In [ ]:
required_paths = [
    "Data",
    "demo_pipeline.py",
    "requirements.txt",
    "run.cmd",
    "run_demo.cmd",
]

checks = []
for item in required_paths:
    path = PROJECT_ROOT / item
    if path.is_dir():
        kind = "folder"
    elif path.is_file():
        kind = "file"
    else:
        kind = "missing"
    checks.append({"path": item, "exists": path.exists(), "type": kind})

checks_df = pd.DataFrame(checks)
display(checks_df)

missing = checks_df.loc[~checks_df["exists"], "path"].tolist()
assert not missing, f"Missing required repository items: {missing}"

## 3. Inspect demo data

The included `Data/` files are short anonymised demo snippets. This step lists the available CSV files and previews one signal table.

In [ ]:
data_files = sorted(DATA_DIR.glob("*.csv"))
assert data_files, "No CSV files were found in Data/."

data_file_table = pd.DataFrame(
    {
        "file": [path.name for path in data_files],
        "size_kb": [round(path.stat().st_size / 1024, 1) for path in data_files],
    }
)
display(data_file_table)

manifest_path = DATA_DIR / "demo_subjects.csv"
if manifest_path.exists():
    print("Subject manifest:")
    display(pd.read_csv(manifest_path))

signal_files = sorted(DATA_DIR.glob("*_demo_signals.csv"))
assert signal_files, "No demo signal CSV files were found in Data/."

preview_path = signal_files[0]
preview = pd.read_csv(preview_path, nrows=8)
print(f"Previewing {rel(preview_path)}")
display(preview)

## 4. Run the demo pipeline

This cell runs the existing pipeline script. It uses the current notebook kernel's Python executable while showing the equivalent command-line form, `python demo_pipeline.py`.

In [ ]:
command_label = "python demo_pipeline.py"
command = [sys.executable, "demo_pipeline.py"]

print(f"Running existing script: {command_label}")
completed = subprocess.run(
    command,
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True,
)

stdout = sanitize_output(completed.stdout)
stderr = sanitize_output(completed.stderr)

if stdout:
    print("stdout:")
    print(tail(stdout))
if stderr:
    print("stderr:")
    print(tail(stderr, lines=40))

assert completed.returncode == 0, f"demo_pipeline.py failed with exit code {completed.returncode}"

## 5. Inspect generated outputs

After a full demo run, the main cohort outputs are written under `Results/cohort_shared/`. This step lists the generated CSV, PNG, and summary files.

In [ ]:
assert RESULTS_DIR.exists(), "Results/ was not created. Run the pipeline cell first."

out_dir = RESULTS_DIR / "cohort_shared"
if not out_dir.exists():
    candidates = [path for path in RESULTS_DIR.iterdir() if path.is_dir()]
    assert candidates, "No result folders were found under Results/."
    out_dir = max(candidates, key=lambda path: path.stat().st_mtime)

print(f"Using output folder: {rel(out_dir)}")

output_files = sorted(
    path for path in out_dir.glob("*")
    if path.is_file() and path.suffix.lower() in {".csv", ".png", ".txt"}
)
assert output_files, f"No CSV, PNG, or TXT outputs were found in {rel(out_dir)}."

output_table = pd.DataFrame(
    {
        "file": [path.name for path in output_files],
        "type": [path.suffix.lower() for path in output_files],
        "size_kb": [round(path.stat().st_size / 1024, 1) for path in output_files],
    }
)
display(output_table)

assert any(path.suffix.lower() == ".csv" for path in output_files), "No generated CSV files were found."
assert any(path.suffix.lower() == ".png" for path in output_files), "No generated PNG files were found."

## 6. Display main figures

The pipeline saves representative figures for the shared embedding, KDE atlas, atlas overlay, and region/activity summaries. The notebook displays whichever of these expected figures are available.

In [ ]:
figure_names = [
    "step3_shared_embedding_by_label.png",
    "step3_shared_embedding_by_group.png",
    "step4_shared_atlas_heatmap.png",
    "step4_shared_atlas_overlay.png",
    "step7_region_activity_circular_network.png",
    "step7b_group_region_barplots_with_deltas.png",
]

figure_paths = [out_dir / name for name in figure_names if (out_dir / name).exists()]
assert figure_paths, "No representative output figures were found."

for path in figure_paths:
    image = plt.imread(path)
    width = 8
    height = max(3, min(6, width * image.shape[0] / image.shape[1]))
    fig, ax = plt.subplots(figsize=(width, height))
    ax.imshow(image)
    ax.set_title(path.name)
    ax.axis("off")
    plt.show()

## 7. Load key CSV outputs

These CSV files expose the main numeric outputs: the shared 2D embedding, the mapping from windows to atlas regions, and occupancy summaries across subjects, labels, and regions.

In [ ]:
csv_names = [
    "step3_shared_embedding_all.csv",
    "step4_shared_window_regions.csv",
    "step5_subject_label_region_occupancy.csv",
    "step5_subject_region_occupancy.csv",
]

loaded_any = False
for name in csv_names:
    path = out_dir / name
    if not path.exists():
        print(f"Missing expected CSV: {name}")
        continue
    df = pd.read_csv(path)
    print(f"{rel(path)}: {df.shape[0]} rows x {df.shape[1]} columns")
    display(df.head(8))
    loaded_any = True

assert loaded_any, "No key CSV outputs were loaded."

## 8. Interpretation notes

- The 2D embedding is not a physical coordinate system.
- Nearby points represent motion windows with similar extracted features.
- Dense regions represent repeated motion states or motion phases in the demo snippets.
- Occupancy summaries can compare motion structure across subjects or groups by showing how often windows fall into each atlas region.

## 9. Reproducibility notes

Equivalent command-line usage from the repository root:

```cmd
run.cmd
run.cmd S1 S13
run_demo.cmd
python demo_pipeline.py
```

`run.cmd` is the Windows one-click entry point and performs dependency checks before running the demo. When started without command-line arguments, it prompts for subject IDs; press Enter at the prompt to run all demo subjects, or enter a space-separated subset such as `S1 S13`. `run_demo.cmd` is a backward-compatible wrapper. The direct Python command runs the existing pipeline script after dependencies are installed.